# A reproducible regression pipeline

You are comparing model-development procedures on a historical benchmark. Predict the numeric target from the provided features. Treat this strictly as an educational benchmark, not a medical decision tool. Your deliverable is a reproducible pipeline and a documented held-out error comparison.

Dataset: scikit-learn diabetes benchmark · 442 observations, 10 numeric features

Reference: https://scikit-learn.org/stable/datasets/toy_dataset.html#diabetes-dataset

Original ML Atlas notebook, MIT-licensed code and CC BY 4.0 explanation. Third-party data retains its own license.


## Environment

Run in Jupyter or Colab. For the scikit-learn projects, install scikit-learn >=1.4 and its dependencies in your own environment. No GPU is needed.


## Plan

1. Load the benchmark and record shape, feature names, missingness and target distribution.
2. Reserve 20% of records as a final test using random_state=42. Explain the limits of a random split for a real deployment.
3. Compare DummyRegressor(strategy='mean') with a StandardScaler + Ridge pipeline. Tune alpha with 5-fold cross-validation on training data only.
4. Evaluate held-out MAE and RMSE. Inspect the ten largest residuals and explain what the metric does not capture.
5. Save configuration and produce a short model card with intended educational use and limitations.


In [ ]:
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

X, y = load_diabetes(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
baseline = DummyRegressor(strategy='mean').fit(X_train, y_train)
search = GridSearchCV(make_pipeline(StandardScaler(), Ridge()), {'ridge__alpha':[0.1,1,10,100]}, cv=5, scoring='neg_mean_absolute_error')
search.fit(X_train, y_train)
print('Shape:', X.shape, 'Train/test:', len(y_train), len(y_test))
print('Selected configuration:', search.best_params_)
for name, model in [('Baseline',baseline),('Ridge',search.best_estimator_)]:
    predictions = model.predict(X_test)
    print(name, 'MAE:', round(mean_absolute_error(y_test,predictions),3), 'RMSE:', round(root_mean_squared_error(y_test,predictions),3))


## Review the result

- [ ] A clean run prints the dataset shape and train/test sizes.
- [ ] Preprocessing is inside the cross-validated pipeline.
- [ ] The report includes baseline and selected-model MAE on identical held-out records.
- [ ] No test labels influence hyperparameter selection.
- [ ] A README explains how to reproduce the result and why it is not a clinical model.


## Extend it

Compare an ensemble on the same training folds. Decide whether any improvement justifies its complexity.

Record what changed, why, and how you evaluated it.
